In [ ]:
%cd /kaggle/working

REPO_BRANCH = "main"

!rm -rf temporal-recsys
!git clone --branch {REPO_BRANCH} --single-branch https://github.com/dshhrv/temporal-recsys.git

%cd /kaggle/working/temporal-recsys


In [ ]:
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from src.lightgcn_model import LightGCN

CONFIG_PATH = Path("configs/lightgcn_ce_ml1m.yaml")

def parse_simple_yaml_value(value):
    value = value.strip()

    if value in {"~", "null", "None"}:
        return None

    if value.lower() == "true":
        return True

    if value.lower() == "false":
        return False

    if value.startswith("[") and value.endswith("]"):
        inner = value[1:-1].strip()

        if not inner:
            return []

        return [parse_simple_yaml_value(part.strip()) for part in inner.split(",")]

    try:
        return int(value)
    except ValueError:
        pass

    try:
        return float(value)
    except ValueError:
        return value.strip('"').strip("'")

def load_simple_yaml(path):
    root = {}
    stack = [(-1, root)]

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].rstrip()

        if not line.strip():
            continue

        indent = len(line) - len(line.lstrip(" "))
        key, value = line.strip().split(":", 1)

        while indent <= stack[-1][0]:
            stack.pop()

        parent = stack[-1][1]

        if value.strip() == "":
            child = {}
            parent[key] = child
            stack.append((indent, child))
        else:
            parent[key] = parse_simple_yaml_value(value)

    return root

config = load_simple_yaml(CONFIG_PATH)

SEED = int(config.get("seed", 42))
USE_GPU = bool(config.get("use_gpu", True))
DEVICE = torch.device("cuda" if USE_GPU and torch.cuda.is_available() else "cpu")

MAX_EPOCHS = int(config["epochs"])
EVAL_STEP = int(config.get("eval_step", 1))
PATIENCE = int(config.get("stopping_step", 7))
VALID_METRIC = config.get("valid_metric", "Recall@10")
TRAIN_BATCH_SIZE = int(config["train_batch_size"])
EVAL_BATCH_SIZE = int(config["eval_batch_size"])
LEARNING_RATE = float(config["learning_rate"])
WEIGHT_DECAY = float(config.get("weight_decay", 0.0))
EMBEDDING_REG_WEIGHT = float(config.get("embedding_reg_weight", 0.0))
EMBEDDING_DIM = int(config["embedding_dim"])
NUM_LAYERS = int(config["num_layers"])
USE_ITEM_BIAS = bool(config.get("use_item_bias", True))
TIME_FIELD = config.get("TIME_FIELD", "time_days")
TEST_GRAPH_MODE = config.get("eval_args", {}).get("test_graph", "train_valid")
MASK_SEEN_ITEMS = bool(config.get("eval_args", {}).get("mask_seen_items", True))

OUTPUT_DIR = Path(config.get("output_dir", "/kaggle/working/lightgcn_ce_results"))
EPOCH_METRICS_PATH = OUTPUT_DIR / config.get("epoch_metrics_name", "lightgcn_ce_epoch_metrics.csv")
TEST_METRICS_PATH = OUTPUT_DIR / config.get("test_metrics_name", "lightgcn_ce_test_metrics.csv")
BEST_CHECKPOINT_PATH = OUTPUT_DIR / config.get("checkpoint_name", "lightgcn_ce_best.pt")
LAST_CHECKPOINT_PATH = OUTPUT_DIR / config.get("last_checkpoint_name", "lightgcn_ce_last.pt")
RUN_CONFIG_PATH = OUTPUT_DIR / "lightgcn_ce_run_config.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")
print(f"Config: {CONFIG_PATH}")


In [ ]:
DATA_DIR = Path(config["data_path"])

if not DATA_DIR.exists() and Path("data/tgn_ml1m").exists():
    DATA_DIR = Path("data/tgn_ml1m")

INTERACTIONS_PATH = DATA_DIR / config.get("interactions_path", "interactions.csv")
META_PATH = DATA_DIR / config.get("meta_path", "meta.json")

if not INTERACTIONS_PATH.exists():
    raise FileNotFoundError(f"Interactions file not found: {INTERACTIONS_PATH}")

metadata = json.loads(META_PATH.read_text(encoding="utf-8"))
interactions = pd.read_csv(INTERACTIONS_PATH)

NUM_USERS = int(metadata["n_users"])
NUM_ITEMS = int(metadata["n_items"])
NUM_NODES = NUM_USERS + NUM_ITEMS

interactions["user_idx"] = interactions["src"].astype(np.int64)
interactions["item_idx"] = (interactions["dst"] - NUM_USERS).astype(np.int64)

if not interactions["user_idx"].between(0, NUM_USERS - 1).all():
    raise RuntimeError("user_idx is out of range")

if not interactions["item_idx"].between(0, NUM_ITEMS - 1).all():
    raise RuntimeError("item_idx is out of range")

sort_columns = [TIME_FIELD]

if "event_id" in interactions.columns:
    sort_columns.append("event_id")

interactions = interactions.sort_values(sort_columns, kind="stable").reset_index(drop=True)
TRAIN = interactions[interactions["split"] == "train"].copy().reset_index(drop=True)
VALID = interactions[interactions["split"] == "valid"].copy().reset_index(drop=True)
TEST = interactions[interactions["split"] == "test"].copy().reset_index(drop=True)

warm_item_set = set(TRAIN["item_idx"].unique().tolist())

if not VALID["item_idx"].isin(warm_item_set).all() or not TEST["item_idx"].isin(warm_item_set).all():
    raise RuntimeError("Cold items remain in valid/test. Regenerate the preprocessed tgn_ml1m dataset.")

print(f"Data dir: {DATA_DIR}")
print(f"Train: {len(TRAIN):,}")
print(f"Valid: {len(VALID):,}")
print(f"Test: {len(TEST):,}")
print(f"Users: {NUM_USERS:,}")
print(f"Catalog: {NUM_ITEMS:,} items")
print(f"Split: {metadata.get('split', {}).get('method', 'global temporal 80/10/10, cold items removed')}")


In [ ]:
def build_norm_adj(frame):
    user_ids = torch.tensor(frame["user_idx"].to_numpy(), dtype=torch.long)
    item_node_ids = torch.tensor(frame["item_idx"].to_numpy(), dtype=torch.long) + NUM_USERS
    row = torch.cat([user_ids, item_node_ids])
    col = torch.cat([item_node_ids, user_ids])
    degree = torch.bincount(row, minlength=NUM_NODES).float().clamp_min(1.0)
    values = degree[row].pow(-0.5) * degree[col].pow(-0.5)
    indices = torch.stack([row, col], dim=0)
    return torch.sparse_coo_tensor(indices, values, (NUM_NODES, NUM_NODES)).coalesce().to(DEVICE)

train_norm_adj = build_norm_adj(TRAIN)
train_valid_frame = pd.concat([TRAIN, VALID], ignore_index=True)
train_valid_norm_adj = build_norm_adj(train_valid_frame)

def make_temporal_batches(frame, batch_size):
    user_ids = torch.tensor(frame["user_idx"].to_numpy(), dtype=torch.long)
    item_ids = torch.tensor(frame["item_idx"].to_numpy(), dtype=torch.long)
    timestamps = torch.tensor(frame[TIME_FIELD].to_numpy(dtype=np.float32), dtype=torch.float32)
    times = timestamps.numpy()
    batches = []
    start = 0

    while start < len(frame):
        end = min(start + batch_size, len(frame))

        while end < len(frame) and times[end] == times[end - 1]:
            end += 1

        batches.append((user_ids[start:end], item_ids[start:end], timestamps[start:end]))
        start = end

    return batches

train_user_ids = torch.tensor(TRAIN["user_idx"].to_numpy(), dtype=torch.long)
train_item_ids = torch.tensor(TRAIN["item_idx"].to_numpy(), dtype=torch.long)
valid_loader = make_temporal_batches(VALID, EVAL_BATCH_SIZE)
test_loader = make_temporal_batches(TEST, EVAL_BATCH_SIZE)
train_seen = [set() for _ in range(NUM_USERS)]

for user_id, item_id in zip(TRAIN["user_idx"].to_numpy(), TRAIN["item_idx"].to_numpy()):
    train_seen[int(user_id)].add(int(item_id))

def copy_seen(seen_items):
    return [items.copy() for items in seen_items]

def add_frame_to_seen(seen_items, frame):
    for user_id, item_id in zip(frame["user_idx"].to_numpy(), frame["item_idx"].to_numpy()):
        seen_items[int(user_id)].add(int(item_id))

def iter_train_batches():
    permutation = torch.randperm(train_user_ids.size(0))

    for start in range(0, train_user_ids.size(0), TRAIN_BATCH_SIZE):
        batch_indices = permutation[start:start + TRAIN_BATCH_SIZE]
        yield train_user_ids[batch_indices].to(DEVICE, non_blocking=True), train_item_ids[batch_indices].to(DEVICE, non_blocking=True)

print(f"Train graph edges: {train_norm_adj._nnz():,}")
print(f"Train+valid graph edges: {train_valid_norm_adj._nnz():,}")
print(f"Valid batches: {len(valid_loader):,}")
print(f"Test batches: {len(test_loader):,}")


In [ ]:
model = LightGCN(NUM_USERS, NUM_ITEMS, embedding_dim=EMBEDDING_DIM, num_layers=NUM_LAYERS, use_item_bias=USE_ITEM_BIAS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

history = []
best_valid_score = -float("inf")
best_epoch = None
bad_epochs = 0

run_config = dict(config)
run_config["resolved_data_dir"] = str(DATA_DIR)
run_config["device"] = str(DEVICE)
run_config["num_users"] = NUM_USERS
run_config["num_items"] = NUM_ITEMS

with open(RUN_CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(run_config, file, ensure_ascii=False, indent=2)

def copy_model_state_dict():
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}

def make_checkpoint(epoch, row, valid_metrics, is_best):
    return {"epoch": epoch, "model_state_dict": copy_model_state_dict(), "optimizer_state_dict": optimizer.state_dict(), "valid_metrics": valid_metrics, "best_valid_score": best_valid_score, "best_epoch": best_epoch, "history": history, "config": run_config, "is_best": is_best, "last_epoch_metrics": row}

def load_checkpoint(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)

print(model)
print(f"Valid metric: {VALID_METRIC}")
print(f"Best checkpoint: {BEST_CHECKPOINT_PATH}")


In [ ]:
TOPK = [int(k) for k in config.get("topk", [5, 10, 20])]
MAX_K = max(TOPK)

def update_metrics(totals, scores, targets):
    top_items = scores.topk(MAX_K, dim=1).indices
    matches = top_items.eq(targets.unsqueeze(1)).float()
    ranks = torch.arange(1, MAX_K + 1, device=scores.device, dtype=scores.dtype).unsqueeze(0)

    for k in TOPK:
        matches_at_k = matches[:, :k]
        ranks_at_k = ranks[:, :k]
        totals[f"Recall@{k}"] += matches_at_k.any(dim=1).float().sum().item()
        totals[f"NDCG@{k}"] += (matches_at_k / torch.log2(ranks_at_k + 1.0)).sum(dim=1).sum().item()
        totals[f"MRR@{k}"] += (matches_at_k / ranks_at_k).sum(dim=1).sum().item()

    totals["count"] += targets.size(0)

def seen_before_current_events(user_ids, item_ids, timestamps, seen_items):
    users = user_ids.detach().cpu().tolist()
    items = item_ids.detach().cpu().tolist()
    times = timestamps.detach().cpu().tolist()
    row_seen = [None] * len(users)
    start = 0

    while start < len(users):
        end = start + 1

        while end < len(users) and times[end] == times[start]:
            end += 1

        for row in range(start, end):
            row_seen[row] = seen_items[users[row]].copy()

        for row in range(start, end):
            seen_items[users[row]].add(items[row])

        start = end

    return row_seen

def mask_unavailable_items(scores, targets, row_seen):
    if not MASK_SEEN_ITEMS:
        return

    target_items = targets.detach().cpu().tolist()

    for row, (target_item_id, seen) in enumerate(zip(target_items, row_seen)):
        blocked = seen - {target_item_id}

        if blocked:
            blocked_ids = torch.tensor(list(blocked), dtype=torch.long, device=DEVICE)
            scores[row, blocked_ids] = -torch.inf

@torch.no_grad()
def evaluate(loader, seen_items, norm_adj):
    model.eval()
    user_embeddings, item_embeddings = model.propagate(norm_adj)
    totals = {"count": 0}

    for k in TOPK:
        totals[f"Recall@{k}"] = 0.0
        totals[f"NDCG@{k}"] = 0.0
        totals[f"MRR@{k}"] = 0.0

    for user_ids_cpu, item_ids_cpu, timestamps_cpu in tqdm(loader, leave=False):
        row_seen = seen_before_current_events(user_ids_cpu, item_ids_cpu, timestamps_cpu, seen_items)
        user_ids = user_ids_cpu.to(DEVICE, non_blocking=True)
        item_ids = item_ids_cpu.to(DEVICE, non_blocking=True)
        scores = model.full_sort_logits(user_ids, user_embeddings, item_embeddings)
        mask_unavailable_items(scores, item_ids, row_seen)
        update_metrics(totals, scores, item_ids)

    return {name: value / totals["count"] for name, value in totals.items() if name != "count"}


In [ ]:
def train_one_epoch(epoch):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    user_embeddings, item_embeddings = model.propagate(train_norm_adj)
    num_examples = train_user_ids.size(0)
    num_batches = (num_examples + TRAIN_BATCH_SIZE - 1) // TRAIN_BATCH_SIZE
    total_loss = 0.0
    total_ce_loss = 0.0
    total_examples = 0

    for batch_index, (user_ids, item_ids) in enumerate(tqdm(iter_train_batches(), total=num_batches, leave=False)):
        logits = model.full_sort_logits(user_ids, user_embeddings, item_embeddings)
        ce_loss = model.ce_loss(logits, item_ids)
        reg_loss = model.embedding_regularization(user_ids, item_ids)
        loss = ce_loss + EMBEDDING_REG_WEIGHT * reg_loss
        scaled_loss = loss * (user_ids.size(0) / num_examples)
        scaled_loss.backward(retain_graph=batch_index < num_batches - 1)
        total_loss += loss.item() * user_ids.size(0)
        total_ce_loss += ce_loss.item() * user_ids.size(0)
        total_examples += user_ids.size(0)

    optimizer.step()
    return total_loss / total_examples, total_ce_loss / total_examples

run_started = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_started = time.perf_counter()
    train_started = time.perf_counter()
    train_loss, train_ce_loss = train_one_epoch(epoch)
    train_seconds = time.perf_counter() - train_started
    validation_started = time.perf_counter()
    valid_metrics = evaluate(valid_loader, copy_seen(train_seen), train_norm_adj)
    validation_seconds = time.perf_counter() - validation_started
    epoch_seconds = time.perf_counter() - epoch_started
    valid_score = valid_metrics[VALID_METRIC]
    is_best = valid_score > best_valid_score

    if is_best:
        best_valid_score = valid_score
        best_epoch = epoch
        bad_epochs = 0
    else:
        bad_epochs += 1

    row = {"epoch": epoch, "train_loss": train_loss, "train_ce_loss": train_ce_loss, **valid_metrics, "is_best": is_best, "best_epoch": best_epoch, "best_valid_score": best_valid_score, "bad_epochs": bad_epochs, "train_seconds": train_seconds, "validation_seconds": validation_seconds, "epoch_seconds": epoch_seconds, "cumulative_seconds": time.perf_counter() - run_started}
    history.append(row)
    pd.DataFrame(history).to_csv(EPOCH_METRICS_PATH, index=False)
    checkpoint = make_checkpoint(epoch, row, valid_metrics, is_best)
    torch.save(checkpoint, LAST_CHECKPOINT_PATH)

    if is_best:
        torch.save(checkpoint, BEST_CHECKPOINT_PATH)

    print(f"Epoch {epoch:02d}/{MAX_EPOCHS} | loss {train_loss:.5f} | ce {train_ce_loss:.5f} | Recall@5 {valid_metrics['Recall@5']:.5f} | Recall@10 {valid_metrics['Recall@10']:.5f} | Recall@20 {valid_metrics['Recall@20']:.5f} | best_epoch {best_epoch} | bad_epochs {bad_epochs}")

    if bad_epochs >= PATIENCE:
        print(f"Early stopping: {VALID_METRIC} did not improve for {PATIENCE} epoch(s).")
        break

history_df = pd.DataFrame(history)
history_df.to_csv(EPOCH_METRICS_PATH, index=False)
print(f"Training finished in {(time.perf_counter() - run_started) / 3600:.2f} h")
print(f"Best epoch: {best_epoch} | best {VALID_METRIC}: {best_valid_score:.6f}")
print(f"Best checkpoint saved to: {BEST_CHECKPOINT_PATH}")
print(f"Last checkpoint saved to: {LAST_CHECKPOINT_PATH}")


In [ ]:
best_checkpoint = load_checkpoint(BEST_CHECKPOINT_PATH)
model.load_state_dict(best_checkpoint["model_state_dict"])

valid_recheck = evaluate(valid_loader, copy_seen(train_seen), train_norm_adj)
print("\nValidation recheck from best checkpoint")
print("Stored best valid:", best_checkpoint["valid_metrics"])
print("Recomputed valid:", valid_recheck)

test_norm_adj = train_valid_norm_adj if TEST_GRAPH_MODE == "train_valid" else train_norm_adj
test_seen = copy_seen(train_seen)
add_frame_to_seen(test_seen, VALID)

test_started = time.perf_counter()
test_metrics = evaluate(test_loader, test_seen, test_norm_adj)
test_seconds = time.perf_counter() - test_started

test_results = pd.DataFrame([{"best_epoch": int(best_checkpoint["epoch"]), f"best_valid_{VALID_METRIC}": float(best_checkpoint["valid_metrics"][VALID_METRIC]), **test_metrics, "test_seconds": test_seconds}])
test_results.to_csv(TEST_METRICS_PATH, index=False)

print("\nTest metrics from best validation checkpoint")
print(test_results)
print(f"Test graph: {TEST_GRAPH_MODE}")
print(f"Test time: {test_seconds / 3600:.2f} h")


In [ ]:
history_df


In [ ]:
test_results


In [ ]:
for path in [EPOCH_METRICS_PATH, TEST_METRICS_PATH, BEST_CHECKPOINT_PATH, LAST_CHECKPOINT_PATH, RUN_CONFIG_PATH]:
    if path.exists():
        print(path)
